# Phase 2: KV Cache Scaling Benchmark on GPU (Kaggle / Colab T4)

This notebook benchmarks our from-scratch **Key-Value (KV) Cache** against Naive autoregressive re-computation and the Hugging Face reference on an NVIDIA Tesla T4 GPU.

### Objectives:
1. **Pre-allocated KV Cache**: Evaluate prefill vs decode separation on GPU.
2. **Scaling Benchmark**: Profile generation throughput and latency across sequence lengths: **128**, **512**, and **2048** tokens.
3. **Memory Isolation**: Measure peak GPU VRAM separately per engine using `torch.cuda.reset_peak_memory_stats()` and explicit cache clearing.
4. **Time Budget**: Safely bound the quadratic Naive 2048 run to a 10-minute budget so it reports partial results without hanging.

In [ ]:
# 1. Environment & Hardware Verification
!nvidia-smi

import torch, transformers, safetensors
print("=" * 60)
print(f"GPU Name:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"PyTorch Version:       {torch.__version__}")
print(f"Transformers Version:  {transformers.__version__}")
print(f"Safetensors Version:   {safetensors.__version__}")
if torch.cuda.is_available():
    print(f"Total GPU Memory:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 60)

In [ ]:
# 2. Clone Repository (phase-2 branch) & Install Package
import os, sys

if not os.path.exists("Mini-LLM"):
    !git clone -b phase-2 https://github.com/mekanH-c/Mini-LLM.git
    %cd Mini-LLM
elif os.path.basename(os.getcwd()) != "Mini-LLM":
    %cd Mini-LLM

!pip install -q -e . --no-deps
!pip install -q matplotlib psutil

sys.path.insert(0, os.path.abspath("src"))

In [ ]:
# 3. Quick Correctness Check on GPU
import gc
from transformers import AutoTokenizer
from mini_infer.loader import load_from_hf
from mini_infer.generate import generate, generate_cached

MODEL_ID = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading custom model on GPU for correctness check...")
model, _ = load_from_hf(MODEL_ID, device=device, dtype=dtype)
model.eval()

check_prompt = "The capital of France is"
out_naive = generate(model, tokenizer, check_prompt, max_new_tokens=20, device=device)
out_cached = generate_cached(model, tokenizer, check_prompt, max_new_tokens=20, device=device)

naive_ids = out_naive.token_ids[0].tolist()
cached_ids = out_cached.token_ids[0].tolist()
assert naive_ids == cached_ids, "Correctness mismatch on GPU!"
print(f"Correctness PASSED on GPU: Generated identical 20 tokens: {out_cached.text[0]!r}")

# Purge GPU memory before starting benchmarks
del model, out_naive, out_cached
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
# 4. Run Phase 2 Benchmark on GPU (Lengths 128, 512, 2048)
# Executes benchmarks/run_kv_cache.py with isolated memory and 10-minute time budget for Naive
!python benchmarks/run_kv_cache.py --lengths 128 512 2048 --device cuda --cached-runs 3 --hf-runs 3 --naive-128-runs 3 --time-budget 600.0 --plot-path benchmarks/kv_cache_scaling_gpu.png

In [ ]:
# 5. Display Scaling Chart
from IPython.display import Image, display
import os

chart_file = "benchmarks/kv_cache_scaling_gpu.png"
if os.path.exists(chart_file):
    display(Image(filename=chart_file))
else:
    print("Chart file not found.")